
# Report-quality analysis: 3D Heisenberg + DMI Monte Carlo

This notebook reads the C++ simulation output from

```text
results/processed/summary.csv
results/processed/summary_by_L_T_mean.csv
results/processed/summary_by_L_T_std.csv
```

and produces report-ready figures for:

- energy density $\langle E\rangle/N$,
- magnetization density $\langle |M|\rangle/N$,
- magnetic susceptibilities,
- heat capacity $C_V/N$,
- helical order parameter $\max_{q\neq 0} S(q)$,
- helical susceptibility-like fluctuations,
- theoretical checks for the helix pitch and low-temperature energy.

The output figures are written to:

```text
figures/report/
```

Run this notebook from either the project root or from the `notebooks/` folder.


In [ ]:

from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Basic, report-friendly matplotlib settings.
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.figsize": (6.2, 4.2),
    "figure.dpi": 120,
    "savefig.bbox": "tight",
})


def find_project_root() -> Path:
    """Find the termproject root folder from the current notebook location."""
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for p in candidates:
        if (p / "results").exists() and (p / "src").exists():
            return p
    # Fallback: if launched from notebooks, use parent; otherwise current directory.
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd


ROOT = find_project_root()
PROCESSED = ROOT / "results" / "processed"
CONFIGS = ROOT / "results" / "configs"
FIG_DIR = ROOT / "figures" / "report"
TABLE_DIR = ROOT / "results" / "processed" / "report_tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Processed results:", PROCESSED)
print("Figures will be written to:", FIG_DIR)



## Load simulation data

`summary.csv` contains one row per independent run, typically one row per `(L, T, seed)`.
The seed-averaged files are useful, but this notebook mostly recomputes means and standard errors directly from `summary.csv`, so that the error bars are consistent.


In [ ]:

summary_path = PROCESSED / "summary.csv"
if not summary_path.exists():
    raise FileNotFoundError(
        f"Could not find {summary_path}. Run first:\n"
        "  python3 scripts/collect_results.py"
    )

raw = pd.read_csv(summary_path)
raw = raw.sort_values(["L", "T", "seed"]).reset_index(drop=True)

print(f"Loaded {len(raw)} runs")
print("Columns:")
print(list(raw.columns))
raw.head()


In [ ]:

# Columns used throughout the analysis.
GROUP_COLS = ["L", "T", "J", "D", "Bx", "By", "Bz"]

# Some simulations might not have all optional columns, so we guard accesses below.
def require_columns(df: pd.DataFrame, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

require_columns(raw, ["L", "T", "J", "D", "seed", "E_density_mean", "M_abs_mean", "Cv_per_spin", "helical_order_mean"])

# Number of seeds per L,T.
seed_counts = raw.groupby(["L", "T"], as_index=False)["seed"].nunique().rename(columns={"seed": "n_seeds"})
seed_counts.head()



## Helper functions

Error bars are computed as the standard error over independent seeds whenever there is more than one seed. If only one seed is available, the notebook omits the seed error bar for observables without a separate within-chain error estimate.


In [ ]:

def grouped_stats(df: pd.DataFrame, y_col: str) -> pd.DataFrame:
    """Mean and standard error over seeds for a chosen observable."""
    require_columns(df, GROUP_COLS + ["seed", y_col])
    g = (
        df.groupby(GROUP_COLS, as_index=False)
        .agg(
            y_mean=(y_col, "mean"),
            y_std=(y_col, "std"),
            n_seeds=("seed", "nunique"),
        )
        .sort_values(["L", "T"])
        .reset_index(drop=True)
    )
    g["y_sem"] = g["y_std"] / np.sqrt(g["n_seeds"])
    return g


def save_figure(fig, filename_base: str):
    """Save both PDF and PNG versions."""
    pdf = FIG_DIR / f"{filename_base}.pdf"
    png = FIG_DIR / f"{filename_base}.png"
    fig.savefig(pdf)
    fig.savefig(png, dpi=250)
    print(f"Saved {pdf}")
    print(f"Saved {png}")


def plot_observable(
    df: pd.DataFrame,
    y_col: str,
    ylabel: str,
    filename_base: str,
    title: str | None = None,
    show_errorbars: bool = True,
    marker: str = "o",
):
    """Plot one observable versus temperature for all available system sizes."""
    stats = grouped_stats(df, y_col)
    fig, ax = plt.subplots()

    for L, g in stats.groupby("L"):
        g = g.sort_values("T")
        yerr = None
        if show_errorbars and g["n_seeds"].max() > 1:
            yerr = g["y_sem"].to_numpy()
        ax.errorbar(
            g["T"],
            g["y_mean"],
            yerr=yerr,
            marker=marker,
            capsize=3 if yerr is not None else 0,
            linewidth=1.6,
            label=fr"$L={int(L)}$",
        )

    ax.set_xlabel(r"Temperature $T$")
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_figure(fig, filename_base)
    return fig, ax, stats


def peak_estimates(df: pd.DataFrame, y_col: str, name: str) -> pd.DataFrame:
    """Estimate pseudo-critical temperatures from the maximum of a response function."""
    stats = grouped_stats(df, y_col)
    rows = []
    for L, g in stats.groupby("L"):
        g = g.sort_values("T")
        idx = g["y_mean"].idxmax()
        row = g.loc[idx]
        rows.append({
            "observable": name,
            "L": int(L),
            "T_peak": float(row["T"]),
            "peak_value": float(row["y_mean"]),
            "n_seeds": int(row["n_seeds"]),
        })
    return pd.DataFrame(rows)


def steepest_drop_estimates(df: pd.DataFrame, y_col: str = "helical_order_mean") -> pd.DataFrame:
    """Estimate transition region from the steepest drop of the helical order parameter."""
    stats = grouped_stats(df, y_col)
    rows = []
    for L, g in stats.groupby("L"):
        g = g.sort_values("T")
        if len(g) < 3:
            continue
        T = g["T"].to_numpy(dtype=float)
        y = g["y_mean"].to_numpy(dtype=float)
        grad = np.gradient(y, T)
        idx = int(np.argmin(grad))
        rows.append({
            "observable": "steepest drop of helical order",
            "L": int(L),
            "T_drop": float(T[idx]),
            "slope": float(grad[idx]),
            "helical_order_at_drop": float(y[idx]),
            "n_seeds": int(g["n_seeds"].iloc[idx]),
        })
    return pd.DataFrame(rows)



## Main observables

These are the direct analogues of the Exercise 01 plots, with the additional helical order parameter suitable for the DMI system.


In [ ]:

fig_energy, ax_energy, energy_stats = plot_observable(
    raw,
    "E_density_mean",
    ylabel=r"Energy density $\langle E\rangle/N$",
    filename_base="energy_density_vs_T_report",
    title="Energy density",
)


In [ ]:

fig_mag, ax_mag, mag_stats = plot_observable(
    raw,
    "M_abs_mean",
    ylabel=r"Magnetization density $\langle |M|\rangle/N$",
    filename_base="magnetization_density_vs_T_report",
    title="Magnetization density",
)


In [ ]:

fig_cv, ax_cv, cv_stats = plot_observable(
    raw,
    "Cv_per_spin",
    ylabel=r"Heat capacity $C_V/N$",
    filename_base="heat_capacity_vs_T_report",
    title="Heat capacity from energy fluctuations",
)


In [ ]:

fig_h, ax_h, helical_stats = plot_observable(
    raw,
    "helical_order_mean",
    ylabel=r"Helical order $\max_{q\neq 0} S(q)$",
    filename_base="helical_order_vs_T_report",
    title="Helical order parameter",
)


In [ ]:

if "chi_abs" in raw.columns:
    fig_chi_abs, ax_chi_abs, chi_abs_stats = plot_observable(
        raw,
        "chi_abs",
        ylabel=r"Magnetic susceptibility $\chi_{abs}$",
        filename_base="susceptibility_abs_vs_T_report",
        title="Magnetic susceptibility from $|M|$ fluctuations",
    )

if "chi_vec" in raw.columns:
    fig_chi_vec, ax_chi_vec, chi_vec_stats = plot_observable(
        raw,
        "chi_vec",
        ylabel=r"Vector susceptibility $\chi_{vec}$",
        filename_base="susceptibility_vec_vs_T_report",
        title="Vector magnetic susceptibility",
    )


In [ ]:

if "helical_susc_like" in raw.columns:
    fig_hs, ax_hs, hs_stats = plot_observable(
        raw,
        "helical_susc_like",
        ylabel=r"Helical susceptibility-like fluctuation",
        filename_base="helical_susc_like_vs_T_report",
        title="Fluctuations of the helical order parameter",
    )



## Pseudo-critical temperature estimates

For finite systems, response peaks are rounded. The peak locations should be interpreted as pseudo-critical temperatures $T_c(L)$, not as the exact thermodynamic-limit critical temperature.


In [ ]:

peak_tables = []
peak_tables.append(peak_estimates(raw, "Cv_per_spin", "heat capacity"))
if "chi_abs" in raw.columns:
    peak_tables.append(peak_estimates(raw, "chi_abs", "chi_abs"))
if "chi_vec" in raw.columns:
    peak_tables.append(peak_estimates(raw, "chi_vec", "chi_vec"))
if "helical_susc_like" in raw.columns:
    peak_tables.append(peak_estimates(raw, "helical_susc_like", "helical_susc_like"))

peaks = pd.concat(peak_tables, ignore_index=True)
drops = steepest_drop_estimates(raw, "helical_order_mean")

peaks.to_csv(TABLE_DIR / "pseudo_critical_peak_estimates.csv", index=False)
drops.to_csv(TABLE_DIR / "helical_order_steepest_drop_estimates.csv", index=False)

print("Peak estimates:")
display(peaks)
print("Steepest drop estimates:")
display(drops)


In [ ]:

# Plot pseudo-critical estimates versus system size, if several L values are available.
if peaks["L"].nunique() > 1:
    fig, ax = plt.subplots()
    for obs, g in peaks.groupby("observable"):
        ax.plot(g["L"], g["T_peak"], marker="o", linewidth=1.6, label=obs)
    if not drops.empty:
        ax.plot(drops["L"], drops["T_drop"], marker="s", linewidth=1.6, label="helical order drop")
    ax.set_xlabel(r"System size $L$")
    ax.set_ylabel(r"Pseudo-critical temperature estimate")
    ax.set_title(r"Finite-size estimates of $T_c(L)$")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, "pseudo_critical_temperature_vs_L_report")
else:
    print("Only one system size available; pseudo-critical vs L plot skipped.")



## Theory checks: helical pitch and low-temperature energy

For a simple helix along one lattice direction, the bond energy along the helix direction is approximately

\[
e(\theta)=-J\cos\theta-D\sin\theta.
\]

Minimization gives

\[
q_0=\theta=\arctan(D/J),
\]

and the corresponding pitch is

\[
\lambda=\frac{2\pi}{q_0}.
\]

For a helix along one axis, a useful ground-state energy estimate is

\[
E_0/N\approx -2J-\sqrt{J^2+D^2}.
\]

The finite lattice only supports momenta $q_n=2\pi n/L$, so the measured peak should be compared with the nearest allowed momentum.


In [ ]:

def equivalent_abs_q(q: float) -> float:
    """Map q in [0, 2*pi) to its absolute equivalent in [0, pi]."""
    twopi = 2.0 * math.pi
    q = q % twopi
    return min(q, twopi - q)


def theory_check_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (L, J, D), g_all in df.groupby(["L", "J", "D"]):
        L = int(L)
        J = float(J)
        D = float(D)
        q0 = math.atan2(D, J)
        pitch = 2.0 * math.pi / q0 if abs(q0) > 1e-14 else np.inf
        n_nearest = int(round(q0 * L / (2.0 * math.pi)))
        n_nearest = max(1, min(L - 1, n_nearest)) if L > 1 else 0
        q_allowed = 2.0 * math.pi * n_nearest / L if L > 0 else np.nan
        E0_est = -2.0 * J - math.sqrt(J * J + D * D)

        # Use the lowest temperature available for the numerical check.
        T_min = g_all["T"].min()
        g_low = g_all[g_all["T"] == T_min]
        E_low = g_low["E_density_mean"].mean()
        helical_low = g_low["helical_order_mean"].mean()

        q_peak_abs = np.nan
        if "q_peak_final" in g_low.columns:
            q_peak_abs = float(np.nanmean([equivalent_abs_q(q) for q in g_low["q_peak_final"].to_numpy()]))

        rows.append({
            "L": L,
            "J": J,
            "D": D,
            "q0_theory": q0,
            "pitch_theory": pitch,
            "nearest_n": n_nearest,
            "nearest_allowed_abs_q": equivalent_abs_q(q_allowed),
            "measured_abs_q_peak_lowT": q_peak_abs,
            "E0_over_N_estimate": E0_est,
            "T_lowest": float(T_min),
            "E_over_N_lowest_T": float(E_low),
            "helical_order_lowest_T": float(helical_low),
        })
    return pd.DataFrame(rows).sort_values(["L", "J", "D"]).reset_index(drop=True)


theory_checks = theory_check_table(raw)
theory_checks.to_csv(TABLE_DIR / "theory_checks_pitch_energy.csv", index=False)
display(theory_checks)


In [ ]:

# Plot measured q_peak against the theoretical q0.
if "q_peak_final" in raw.columns:
    fig, ax = plt.subplots()
    for L, g in raw.groupby("L"):
        g = g.copy()
        g["abs_q_peak"] = g["q_peak_final"].apply(equivalent_abs_q)
        q_stats = g.groupby("T", as_index=False).agg(
            q_mean=("abs_q_peak", "mean"),
            q_std=("abs_q_peak", "std"),
            n=("seed", "nunique"),
        )
        q_stats["q_sem"] = q_stats["q_std"] / np.sqrt(q_stats["n"])
        yerr = q_stats["q_sem"].to_numpy() if q_stats["n"].max() > 1 else None
        ax.errorbar(q_stats["T"], q_stats["q_mean"], yerr=yerr, marker="o", capsize=3, label=fr"$L={int(L)}$")

    # Add theoretical q0 line for the first J,D pair.
    J0 = float(raw["J"].iloc[0])
    D0 = float(raw["D"].iloc[0])
    q0 = math.atan2(D0, J0)
    ax.axhline(q0, linestyle="--", linewidth=1.4, label=fr"$q_0=\arctan(D/J)={q0:.3f}$")
    ax.set_xlabel(r"Temperature $T$")
    ax.set_ylabel(r"Dominant $|q|$")
    ax.set_title("Dominant helical wavevector")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, "dominant_q_vs_T_report")



## Acceptance rate diagnostic

The proposal width should not be too small. A very high acceptance rate means the spin proposals are conservative; a very low acceptance rate means most proposed moves are rejected. For production, a rough target around $0.4$--$0.7$ is often reasonable, but the optimal value depends on the temperature and observable autocorrelation.


In [ ]:

if "acceptance_rate" in raw.columns:
    fig_acc, ax_acc, acc_stats = plot_observable(
        raw,
        "acceptance_rate",
        ylabel="Acceptance rate",
        filename_base="acceptance_rate_vs_T_report",
        title="Metropolis acceptance rate",
    )



## Optional: visualize one saved spin configuration

This works if you ran a simulation with `--save-config`. The C++ code writes files like:

```text
results/configs/config_L16_T0p700_J1p000_D0p700_seed1.csv
```

The plot below shows a single 2D slice of the 3D spin field. The arrows show two selected spin components; the color shows a third component.


In [ ]:

def load_config_csv(path: Path) -> np.ndarray:
    """Load a saved configuration CSV into an array of shape (L,L,L,3)."""
    cfg = pd.read_csv(path)
    require_columns(cfg, ["x", "y", "z", "Sx", "Sy", "Sz"])
    L = int(cfg[["x", "y", "z"]].to_numpy().max()) + 1
    spins = np.zeros((L, L, L, 3), dtype=float)
    for row in cfg.itertuples(index=False):
        spins[int(row.x), int(row.y), int(row.z), :] = [row.Sx, row.Sy, row.Sz]
    return spins


def plot_spin_slice(
    spins: np.ndarray,
    plane: str = "xz",
    index: int | None = None,
    stride: int = 1,
    spin_components: tuple[int, int] = (0, 1),
    color_component: int = 2,
    filename_base: str = "spin_slice_report",
):
    L = spins.shape[0]
    if index is None:
        index = L // 2

    if plane == "xy":
        data = spins[:, :, index, :]
        xlabel, ylabel = "x", "y"
    elif plane == "xz":
        data = spins[:, index, :, :]
        xlabel, ylabel = "x", "z"
    elif plane == "yz":
        data = spins[index, :, :, :]
        xlabel, ylabel = "y", "z"
    else:
        raise ValueError("plane must be one of 'xy', 'xz', 'yz'")

    X, Y = np.meshgrid(np.arange(L), np.arange(L), indexing="ij")
    U = data[:, :, spin_components[0]]
    V = data[:, :, spin_components[1]]
    C = data[:, :, color_component]

    fig, ax = plt.subplots(figsize=(6.0, 5.2))
    q = ax.quiver(
        X[::stride, ::stride],
        Y[::stride, ::stride],
        U[::stride, ::stride],
        V[::stride, ::stride],
        C[::stride, ::stride],
        angles="xy",
        scale_units="xy",
        scale=1.5,
        width=0.005,
    )
    cb = fig.colorbar(q, ax=ax)
    cb.set_label(fr"$S_{['x','y','z'][color_component]}$")
    ax.set_aspect("equal")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(fr"Spin slice: {plane}, index={index}")
    fig.tight_layout()
    save_figure(fig, filename_base)
    return fig, ax


config_files = sorted(CONFIGS.glob("config_*.csv"))
print(f"Found {len(config_files)} saved configuration files.")
if config_files:
    print("Using:", config_files[0])
    spins = load_config_csv(config_files[0])
    _ = plot_spin_slice(spins, plane="xz", index=spins.shape[0]//2, stride=1, filename_base="spin_slice_example_report")
else:
    print("No saved configurations found. Run one simulation with --save-config to use this cell.")



## Suggested text snippets for the report

Use these only after checking that the plots support them for the final production data.

- The total magnetization remains small because the ordered state is helical rather than ferromagnetic, so the spatially rotating spins largely cancel in the total magnetization.
- The nonzero-wave-vector structure factor is therefore a more appropriate order parameter than $|M|/N$.
- The low-temperature dominant wavevector is consistent with the theoretical estimate $q_0=\arctan(D/J)$ up to the discretization of allowed finite-size momenta $q_n=2\pi n/L$.
- The heat capacity and the helical-order fluctuations show rounded finite-size peaks. Their peak locations are interpreted as pseudo-critical temperatures $T_c(L)$.
- Larger lattices should sharpen the response peaks and make the drop of the helical order parameter more abrupt.
